# 07 — Verify All Scenarios

## Goal
Run the **complete detection pipeline** on real USDJPY=X data across all 4 timeframes and verify that:

1. All 4 formation types (**RBR, DBD, DBR, RBD**) appear in the data
2. Every zone object contains all expected fields (proximal, distal, departure, passed)
3. The pass/reject split looks reasonable per timeframe

This is the end-to-end integration check before we move on to scoring features.

---

## Pipeline recap

```
raw OHLCV
    │
    ▼
CandlePrimitives.enrich_dataframe()  +  add_atr()
    │
    ▼
detect_bases()          →  base clusters (width, compactness gates)
    │
    ▼
detect_formations()     →  leg measurement → RBR / DBD / DBR / RBD
    │
    ▼
detect_zones()          →  proximal / distal + departure gates
    │
    ▼
passed_zones  /  rejected_zones
```

## 1. Setup & load data

In [6]:
import sys
sys.path.append("..")

import pandas as pd
from IPython.display import display

from utils.data_loader import load_all_timeframes
from utils.models import CandlePrimitives, add_atr
from utils.base_detector import detect_bases
from utils.legs_formation import detect_formations
from utils.zone_detector import detect_zones
from utils.config import (
    COLOR_BULL, COLOR_BEAR,
    CHART_BG as BG, CHART_GRID as GRID,
    DEPARTURE_CANDLES,
)

SYMBOL = "USDJPY=X"

raw  = load_all_timeframes(SYMBOL, align=True)
data = {tf: add_atr(CandlePrimitives.enrich_dataframe(df)) for tf, df in raw.items()}

print(f"Loaded {len(data)} timeframes: {list(data.keys())}")
for tf, df in data.items():
    print(f"  {tf:4s}  {len(df):5d} rows  {df.index[0].date()} → {df.index[-1].date()}")

Loaded 4 timeframes: ['1wk', '1d', '4h', '1h']
  1wk     104 rows  2024-06-09 → 2026-05-31
  1d      517 rows  2024-06-04 → 2026-06-03
  4h     3175 rows  2024-06-04 → 2026-06-04
  1h    12262 rows  2024-06-04 → 2026-06-04


## 2. Run the full pipeline

In [7]:
# bases → formations → zones for all 4 timeframes
results = {}
for tf, df in data.items():
    passed_bases, _   = detect_bases(df)
    formations        = detect_formations(df, passed_bases)
    passed_zones, rej = detect_zones(df, formations)
    results[tf] = {
        "df":           df,
        "formations":   formations,
        "passed_zones": passed_zones,
        "rejected":     rej,
    }

# Summary table — one row per timeframe
rows = []
for tf, r in results.items():
    pz = r["passed_zones"]
    rj = r["rejected"]
    formation_counts = {}
    for z in pz:
        formation_counts[z["formation"]] = formation_counts.get(z["formation"], 0) + 1
    rows.append({
        "timeframe":     tf,
        "formations_in": len(r["formations"]),
        "zones_passed":  len(pz),
        "RBR":           formation_counts.get("RBR", 0),
        "DBR":           formation_counts.get("DBR", 0),
        "DBD":           formation_counts.get("DBD", 0),
        "RBD":           formation_counts.get("RBD", 0),
        "rejected_dep":  len(rj),
        "pass_rate":     f"{len(pz)/len(r['formations']):.0%}" if r["formations"] else "—",
    })

summary = pd.DataFrame(rows).set_index("timeframe")
display(summary)

,formations_in,zones_passed,RBR,DBR,DBD,RBD,rejected_dep,pass_rate
timeframe,,,,,,,,
1wk,2,0,0,0,0,0,2,0%
1d,1,0,0,0,0,0,1,0%
4h,33,25,5,5,9,6,8,76%
1h,135,97,22,21,27,27,38,72%


## 3. Verify all 4 formation types exist

In [8]:
# Collect all passed zones across all timeframes
all_passed = []
for tf, r in results.items():
    for z in r["passed_zones"]:
        all_passed.append({**z, "timeframe": tf})

all_formations = {z["formation"] for z in all_passed}
EXPECTED_FORMATIONS = {"RBR", "DBD", "DBR", "RBD"}

print("Formation coverage check")
print("-" * 40)
for f in sorted(EXPECTED_FORMATIONS):
    count = sum(1 for z in all_passed if z["formation"] == f)
    zone_type = "demand" if f in ("RBR", "DBR") else "supply"
    status = "✓" if f in all_formations else "✗ NOT FOUND"
    print(f"  {f}  ({zone_type:6s})  {count:3d} zones   {status}")

print()
missing = EXPECTED_FORMATIONS - all_formations
if missing:
    print(f"⚠ Missing formations: {missing}")
else:
    print("✓ All 4 formation types found across the dataset")

Formation coverage check
----------------------------------------
  DBD  (supply)   36 zones   ✓
  DBR  (demand)   26 zones   ✓
  RBD  (supply)   33 zones   ✓
  RBR  (demand)   27 zones   ✓

✓ All 4 formation types found across the dataset


## 4. Inspect — daily zones detail

In [9]:
TF = "1d"
df_1d = results[TF]["df"]

detail_rows = []
for z in results[TF]["passed_zones"] + results[TF]["rejected"]:
    bs, be = z["start"], z["end"]
    detail_rows.append({
        "date_start": df_1d.index[bs].date(),
        "date_end":   df_1d.index[be].date(),
        "formation":  z["formation"],
        "zone_type":  z["zone_type"],
        "proximal":   round(z["proximal"], 3),
        "distal":     round(z["distal"],   3),
        "zone_width": round(z["zone_width"], 3),
        "dep_atr":    z["dep_atr"],
        "dep_ratio":  z["dep_ratio"],
        "passed":     z["passed"],
    })

detail = pd.DataFrame(detail_rows).sort_values("date_start")
if detail.empty:
    print(f"No zones found for {TF}")
else:
    display(detail.to_string(index=False))

'date_start   date_end formation zone_type  proximal  distal  zone_width  dep_atr  dep_ratio  passed\n2026-02-04 2026-02-10       RBD    supply   154.074 157.328       3.254    1.211      0.552   False'

## 5. Visualize — all 4 timeframes

In [10]:
import plotly.graph_objects as go

PLOT_MONTHS = 36

FILL_PASSED = {"demand": "rgba(38,166,154,0.15)", "supply": "rgba(239,83,80,0.15)"}
FILL_REJ    = {"demand": "rgba(38,166,154,0.06)", "supply": "rgba(239,83,80,0.06)"}
EDGE_COLOR  = {"demand": "#26a69a",               "supply": "#ef5350"}


def plot_zones(tf: str, r: dict) -> None:
    df        = r["df"]
    all_zones = r["passed_zones"] + r["rejected"]

    cutoff  = df.index[-1] - pd.DateOffset(months=PLOT_MONTHS)
    df_plot = df[df.index >= cutoff].copy()
    offset  = len(df) - len(df_plot)

    vis_zones    = [z for z in all_zones if z["end"] >= offset]
    n_passed_all = len(r["passed_zones"])
    n_rej_all    = len(r["rejected"])

    fig = go.Figure()

    fig.add_trace(go.Candlestick(
        x=df_plot.index,
        open=df_plot["open"], high=df_plot["high"],
        low=df_plot["low"],   close=df_plot["close"],
        increasing_line_color=COLOR_BULL,
        decreasing_line_color=COLOR_BEAR,
        name="Price",
    ))

    for z in vis_zones:
        bs_plot = max(z["start"] - offset, 0)
        be_plot = min(z["end"]   - offset, len(df_plot) - 1)
        zt      = z["zone_type"]
        passed  = z["passed"]

        x0 = df_plot.index[bs_plot]
        x1 = df_plot.index[be_plot]

        fig.add_vrect(
            x0=x0, x1=x1,
            fillcolor=FILL_PASSED[zt] if passed else FILL_REJ[zt],
            line_color=EDGE_COLOR[zt],
            line_width=1,
            line_dash="solid" if passed else "dot",
            opacity=1.0,
        )

        ext_iloc = min(z["end"] - offset + DEPARTURE_CANDLES + 1, len(df_plot) - 1)
        x_ext = df_plot.index[ext_iloc]

        fig.add_shape(
            type="line", x0=x0, x1=x_ext,
            y0=z["proximal"], y1=z["proximal"],
            line=dict(color=EDGE_COLOR[zt], width=1, dash="dash"),
        )
        fig.add_shape(
            type="line", x0=x0, x1=x_ext,
            y0=z["distal"], y1=z["distal"],
            line=dict(color=EDGE_COLOR[zt], width=1, dash="dot"),
        )

        status = "✓" if passed else "✗"
        fig.add_annotation(
            x=x0, y=z["proximal"],
            text=f"{z['formation']} {status}<br>atr={z['dep_atr']} r={z['dep_ratio']}",
            showarrow=False,
            font=dict(size=8, color=EDGE_COLOR[zt]),
            xanchor="left", yanchor="bottom",
        )

    fig.update_layout(
        title=(
            f"{SYMBOL} {tf} — All zones (last {PLOT_MONTHS} months shown)  "
            f"full history: {n_passed_all} passed, {n_rej_all} rejected by departure"
        ),
        xaxis_rangeslider_visible=False,
        template="plotly_dark",
        paper_bgcolor=BG,
        plot_bgcolor=BG,
        xaxis=dict(gridcolor=GRID),
        yaxis=dict(gridcolor=GRID, title="Price"),
        height=500,
    )
    fig.show()


for tf, r in results.items():
    plot_zones(tf, r)

## 6. What's next

| Notebook | Topic | Status |
|---|---|---|
| `08_freshness.ipynb` | Zone freshness — has price already touched the zone? | ⏳ next |

The pipeline is now fully verified on real data. Every zone carries:
- **Formation** (RBR / DBD / DBR / RBD) and zone type
- **Proximal / distal** boundary levels
- **Departure** confirmation

NB08 adds a freshness filter: a zone that has already been tested (price returned to it) is weaker than a fresh, untouched zone.